In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "buttelmann2008behavioral")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "EGG_Study_Big_File__Baseline-G_.sav")
# complete_path_2 = os.path.join(original_data_pathway, "egg_study_results_working_file__single_sessions_.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)

df['test'].unique()
experiment_list=[['baseline','1', 'baseline'], 
                ['EGG Study a','1', 'smell_vs_smell_and_bite'], 
                ['EGG Study b','1', 'shake_vs_shake_and_bite'], 
                ['EGG Study c','1', 'bite_and_smell_vs_smell_and_bite'] ,
                ['EGG Study d','2', 'smell_vs_smell_and_pull_apart'],
                ['EGG Study e','2', 'smell_vs_smell_and_egg_on_head'], 
                ['EGG Study f','3', 'smell_vs_smell_and_bite_empty_eggs'], 
                ['Egg Study g','3', 'egg_at_chest_vs_smell_and_bite']]
for x,y,a in experiment_list:
    df.loc[df.test == x, ['experiment','condition']] = y, a


In [3]:

df['study_id']="buttelmann2008behavioral"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


df.rename(columns={"name": "participant",
                "trialnr":'trial',
                "sidefood":"baited_egg",
                "species":"species_original",
                'sidstart':'side_start',
                "choicsub":"choice",
                "subcorre":"correct"}, inplace=True)


comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

df['side_start'].unique()
df['side_start'].replace('neither... nor...', 'neither', inplace=True)
# df.columns

In [4]:
fulldf = df[['study_id','experiment', 'participant', 'sex', 'species',
'session','trial',  'condition',
      #  'sheetnr',  
       'side_start', 'baited_egg', 'choice', 'correct',
         ]]
for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'buttelmann2008behavioral_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'buttelmann2008behavioral_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)